<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L03-cost-and-quality-decision-calculator/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L03-cost-and-quality-decision-calculator/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L03-cost-and-quality-decision-calculator/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer. The cell below fetches the files it needs beside it,
and does nothing where they are already present. On Kaggle, switch Internet on in the
notebook's settings first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["assets/SOURCE.md", "assets/price_snapshot.json", "assets/scenario.json"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L03-cost-and-quality-decision-calculator/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# Build a cost and quality decision calculator that prices prompting, retrieval, fine-tuning, distillation and a mix over a year, and finds the volumes at which each overtakes the last

**You will build:** a calculator that reads a dated price snapshot and a task description, prices five ways of
getting the task done over a year in three currencies, finds the monthly volumes at which one strategy overtakes
another, and says which strategy is cheapest among those that meet the quality, latency, data and capacity
constraints.
**Time:** ~45 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, offline · **Prerequisites:** T00-L01-the-8gb-track

By the end you will be able to:
1. Implement `convert()` so an amount moves between EUR, USD and INR through the ECB's per-euro reference rates, in the right direction.
2. Implement `request_cost_usd()` and `fixed_costs_usd()` so five strategies are priced per request and per month, with cache hits, a minimum charge, labelling, retraining and an always-on endpoint.
3. Implement `breakeven_volume()` and `ladder()` so the cheapest strategy is a function of volume, and dominated strategies drop out.
4. Measure, on the snapshot, which strategy wins, at what volumes the winner changes, and which price moves change the decision.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write Python functions that take dictionaries and lists and return numbers or lists, with loops and conditionals
- solve a linear equation in one unknown, such as 120 + 6x = 12x, and check the answer by substituting it back
- say what a token is and why model APIs quote prices per million tokens

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Patterns for Building LLM-based Systems & Products](https://eugeneyan.com/writing/llm-patterns/) · Eugene Yan · *free* — a practitioner's tour of fine-tuning, retrieval, caching and evaluation as separate levers, most of the strategies this lesson prices
- [Prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) · Anthropic (Claude Platform docs) · *free* — how a repeated instruction is cached and billed at a fraction of the input price, the mechanism behind the lesson's prompting cost
- [Tinker documentation, Models & Pricing](https://tinker-docs.thinkingmachines.ai/tinker/models/models_and_pricing/) · Thinking Machines Lab · *free* — the page where a managed training service lists its per-million-token training prices, which is how the lesson prices training the distilled small model

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import copy
import gc
import io
import json
import os
import signal
import sys
import threading
import time
import traceback
from pathlib import Path
from typing import Callable

print("python", sys.version.split()[0])




def _guard(what: str, seconds: float = 2):
    """Run a check or a demo under `_time_limit`, so a loop of yours that never ends fails it with a
    message naming `what` instead of freezing the notebook. A check calls your code on tiny inputs, so
    two seconds is plenty, and a loop that appends as it spins can fill gigabytes in a few more. Each run
    first collects what an earlier stopped loop left behind: the frames of a stopped call stay in a
    reference cycle, with everything they built, until the garbage collector frees them."""
    def wrap(fn: Callable[[], None]) -> Callable[[], None]:
        def guarded() -> None:
            gc.collect()
            with _time_limit(seconds, what):
                fn()
        guarded.__name__ = fn.__name__
        return guarded
    return wrap


def asset_path(name: str) -> Path:
    """Locate a file that ships in this lesson's assets/ folder. There is no network branch."""
    try:
        here = Path(__file__).resolve().parent
    except NameError:  # a notebook has no __file__
        here = Path.cwd()
    for candidate in (here / "assets" / name, here.parent / "assets" / name,
                      Path.cwd() / "assets" / name):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"{name} not found in assets/ beside this lesson; it is never downloaded "
                            "by the lesson code (the launcher cell fetches it when it is missing).")


SNAPSHOT: dict = json.loads(asset_path("price_snapshot.json").read_text())
SCENARIO: dict = json.loads(asset_path("scenario.json").read_text())
STRATEGIES = ("prompting", "retrieval", "finetune", "distillation", "mix")
CURRENCIES = tuple(SNAPSHOT["fx"]["units_per_eur"])   # the three currencies the snapshot carries
MTOK = 1_000_000   # prices are quoted per million tokens

print(f"price snapshot dated {SNAPSHOT['snapshot_date']}; exchange rates dated {SNAPSHOT['fx']['rate_date']}")
print("nothing in this notebook reads the clock or the network: every figure below comes from the two files in assets/")

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("convert",),
    "exercise 2": ("request_cost_usd",),
    "exercise 3": ("fixed_costs_usd",),
    "exercise 4": ("breakeven_volume",),
    "exercise 5": ("ladder",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (fixed_costs_usd)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 10     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped: needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped: needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet: fill in {stub}() above, then re-run this cell.")
        else:
            print(f"{label}: skipped: {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED: {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. A decision is a structure; a price is an input

Teams ask "should we fine-tune?" as if it had one answer. It does not. Five ways of getting a task done
differ in **where the money goes**, and that is what decides which is cheapest at a given volume:

- **prompting**: pay per call on a large hosted model, with a long instruction that a cache makes cheaper;
- **retrieval**: the same, plus retrieved passages in every call and a one-off cost of indexing a corpus;
- **finetune**: pay people to label examples, train a small model on them (and again when the data drifts),
  then pay for the machine that serves it, whether or not it is busy;
- **distillation**: let a large open-weights *teacher* model label unlabelled inputs instead, train the same
  small model on its answers, and serve it on the same kind of machine;
- **mix**: serve the distilled small model for the easy share of traffic and send the rest to the hosted model.

A strategy with a big one-off cost and a small cost per request loses at low volume and wins at high volume.
The volume where it starts winning is its **break-even**. Prices move every quarter; this structure does not.

Why does a small model need a machine of its own? On the snapshot date the hosts cited here either serve a
fine-tuned model only on a dedicated GPU, billed for the time it runs, or price per-token sampling of it only as
part of a training service whose per-token serving is a beta for other models (the next cell prints what each
says). So both small models are priced on an always-on GPU: an **assumption** about the deployment, set in
`scenario.json`.

One thing no price table shows: a teacher's licence, and the terms of the service you call it through, must
permit training another model on its answers; the teacher here is released under the Apache 2.0 licence
(`claims.yaml` quotes it), and nothing in this lesson is legal advice.

The prices come from a **committed snapshot**, so the notebook is the same offline on any day. The cell
below prints what it holds and the day each price was read. Quality and latency are not prices: they are
*assumptions* written into `scenario.json` by the lesson's author, and the scenario cell further down says so again.

In [ ]:
print(f"price snapshot dated {SNAPSHOT['snapshot_date']}. Every price below is USD per million tokens unless it says otherwise,")
print("and each line ends with the day it was read. Prices change every quarter: read them as a dated snapshot, never as today's.\n")
_model = SNAPSHOT["api_models"][SCENARIO["prompt_model"]]
print(f"  hosted large model   {SCENARIO['prompt_model']:<24} in {_model['input_usd_per_mtok']}  out {_model['output_usd_per_mtok']}  "
      f"cached input {_model['cache_read_usd_per_mtok']}   read {_model['read_on']}")
_teacher = SNAPSHOT["open_models"][SCENARIO["teacher_model"]]
print(f"  open teacher         {SCENARIO['teacher_model']:<24} in {_teacher['input_usd_per_mtok']}  out {_teacher['output_usd_per_mtok']}  "
      f"on {_teacher['provider']}, licence {_teacher['licence']}   read {_teacher['read_on']}")
_student = SNAPSHOT["tinker"]["models"][SCENARIO["student_model"]]
print(f"  small model, Tinker  {SCENARIO['student_model']:<24} train {_student['train_usd_per_mtok']}   read {SNAPSHOT['tinker']['read_on']}")
print(f"  checkpoint storage   Tinker, USD per GB-month {SNAPSHOT['tinker']['storage_usd_per_gb_month']}   read {SNAPSHOT['tinker']['read_on']}")
_sft = SNAPSHOT["together"]["lora_sft"][SCENARIO["tuned_base"]]
print(f"  LoRA fine-tune       Together, {SCENARIO['tuned_base']}: {_sft['usd_per_mtok']} per Mtok trained, minimum charge "
      f"USD {_sft['minimum_charge_usd']} per job   read {SNAPSHOT['together']['read_on']}")
print(f"  dedicated GPU        Together, {SCENARIO['gpu']}: USD {SNAPSHOT['together']['dedicated_usd_per_gpu_hour'][SCENARIO['gpu']]} "
      f"per GPU-hour, on demand   read {SNAPSHOT['together']['read_on']}")
_other = SNAPSHOT["together"]["dedicated_docs_table"][SCENARIO["gpu"]]
print(f"  {'':<21}(Together's dedicated-inference docs listed USD {_other['promotion_usd_per_gpu_hour']} in their hardware table while a "
      f"promotion ran until {_other['promotion_ended']}; they list USD {_other['usd_per_gpu_hour']} again;\n  {'':<21} the snapshot keeps "
      f"the on-demand price: claims.yaml has both)   read {_other['read_on']}")
print(f"  embeddings           Fireworks, {SNAPSHOT['fireworks']['embedding_applies_to']}: "
      f"{SNAPSHOT['fireworks']['embedding_usd_per_mtok']} per Mtok   read {SNAPSHOT['fireworks']['read_on']}")
_serving = SNAPSHOT["serving_a_fine_tune"]
print(f"\nhow the cited hosts serve a fine-tuned model (paraphrased in the snapshot from their documentation, which claims.yaml "
      f"quotes; read {_serving['read_on']}):")
for _host, _how in _serving.items():
    if _host != "read_on":
        print(f"  {_host + ':':<14}{_how}")
print("so each small model is priced on its own always-on dedicated GPU, an ASSUMPTION about the deployment set in")
print("scenario.json: the distilled one is trained on Tinker, then downloaded and served on such a GPU (claims.yaml);")
print("the open teacher is called per token, like the hosted model")
print("\nexchange rates, units per euro, ECB reference rates dated", SNAPSHOT["fx"]["rate_date"])
for _cur, _rate in SNAPSHOT["fx"]["units_per_eur"].items():
    print(f"  {_cur}  {_rate}")

### The scenario

The task is an imaginary support-ticket triage. The file holds the volume, the size of a request, how much
labelled data exists, how often the data drifts, and what the business needs. **Read every line: each is an
assumption you can change.**

In [ ]:
print(SCENARIO["name"])
print(SCENARIO["authored"], "\n")
for _key, _val in SCENARIO.items():
    if _key not in ("name", "authored", "assumed_quality", "assumed_latency_ms"):
        print(f"  {_key:<38} {_val}")
print("\n  assumed quality, a share of tickets handled correctly (an assumption, not a measurement of any model):")
print("   ", SCENARIO["assumed_quality"])
print("  assumed latency in milliseconds (an assumption):")
print("   ", SCENARIO["assumed_latency_ms"])

## 2. Exercise 1: money across currencies

The ECB publishes **how many units of each currency one euro buys**. The snapshot's `units_per_eur` holds
`EUR: 1.0` and one number for each other currency. To turn an amount in `src` into `dst` you pass through
the euro: first from `src` to euros, then from euros to `dst`.

The common failure is multiplying where you should divide, which gives a plausible-looking number that is
wrong by the square of the rate. The public check below uses made-up currencies so that the right answer
is obvious from the direction alone. An unknown currency is a `ValueError` naming it, not a `KeyError` from
deep inside a dictionary. A negative amount (a refund) converts the same way as a positive one.

<details><summary>💡 Hint 1: what to think about</summary>

If one euro buys many units of a currency, is one unit of that currency worth more or less than a euro?
So when `src` is the currency with the big number, should the amount become bigger or smaller on the way
to euros?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Check both currency codes are in the table. Divide the amount by the source's units-per-euro to get euros,
then multiply by the target's units-per-euro. The same currency on both sides should hand the amount back.
</details>

In [ ]:
def usd(tokens: float, usd_per_mtok: float) -> float:
    """What `tokens` tokens cost at a price quoted in USD per million tokens. Given, not an exercise."""
    return tokens * usd_per_mtok / MTOK


def convert(amount: float, src: str, dst: str, units_per_eur: dict) -> float:
    """Convert `amount` from currency `src` into `dst`, through the euro.

    Example (invented rates: one euro buys 2 AAA and 8 BBB):
        >>> convert(10, "AAA", "BBB", {"EUR": 1.0, "AAA": 2.0, "BBB": 8.0})
        40.0

    Returns a float. Raises ValueError, naming the currency, when `src` or `dst` is not in the table.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks: run them as often as you like.
@_guard("convert()")
def _check_convert() -> None:
    rates = {"EUR": 1.0, "AAA": 2.0, "BBB": 8.0}
    assert abs(convert(10, "AAA", "BBB", rates) - 40) < 1e-9, \
        "10 AAA = 5 EUR = 40 BBB: divide by the source's units per euro, then multiply by the target's"
    assert abs(convert(10, "BBB", "AAA", rates) - 2.5) < 1e-9, \
        "going to the currency with the smaller number must make the amount smaller: check which rate you divide by"
    assert abs(convert(3, "EUR", "BBB", rates) - 24) < 1e-9, "euros to BBB: multiply by BBB's units per euro"
    assert abs(convert(24, "BBB", "EUR", rates) - 3) < 1e-9, "BBB to euros: divide by BBB's units per euro"
    assert abs(convert(7, "AAA", "AAA", rates) - 7) < 1e-9, "the same currency on both sides should not change the amount"
    for bad in (("ZZZ", "AAA"), ("AAA", "ZZZ")):
        try:
            convert(1, bad[0], bad[1], rates)
        except ValueError as exc:
            assert "ZZZ" in str(exc), "the ValueError should name the unknown currency"
        else:
            raise AssertionError(f"{bad} contains an unknown currency: raise ValueError")
    print("exercise 1 looks right")


_try("exercise 1", _check_convert)

### The same price in three currencies

With `convert()` working, the snapshot's prices can be shown in all three currencies. The prices are quoted
in USD, so every row goes USD to the other two.

In [ ]:
@_guard("the price table, which calls convert()", 10)
def _show_currencies() -> None:
    rates = SNAPSHOT["fx"]["units_per_eur"]
    rows = [("hosted large model, input", _model["input_usd_per_mtok"]),
            ("hosted large model, output", _model["output_usd_per_mtok"]),
            ("open teacher, output", _teacher["output_usd_per_mtok"]),
            ("small model, train", _student["train_usd_per_mtok"]),
            ("dedicated GPU, per hour", SNAPSHOT["together"]["dedicated_usd_per_gpu_hour"][SCENARIO["gpu"]])]
    print(f"{'price per million tokens (or per hour)':<40}" + "".join(f"{c:>12}" for c in CURRENCIES))
    for name, price in rows:
        print(f"{name:<40}" + "".join(f"{convert(price, 'USD', c, rates):>12.4f}" for c in CURRENCIES))
    print(f"\none euro buys {convert(1, 'EUR', 'INR', rates):.4f} INR and {convert(1, 'EUR', 'USD', rates):.4f} USD, "
          f"so one US dollar buys {convert(1, 'USD', 'INR', rates):.4f} INR")
    print("the ECB publishes these rates for information; it discourages using them for transactions (claims.yaml)")


_try("price table", _show_currencies, needs=("exercise 1",))

## 3. Exercise 2: what one request costs

`request_cost_usd(strategy, scn, snap)` is the **variable** cost: what each extra request adds. Inputs come
from the scenario `scn` and the snapshot `snap`. Per strategy, in USD:

- `prompting`: on `scn["prompt_model"]`, the instruction, the ticket and the answer. A share
  `cache_hit_share` of the instruction tokens is a cache hit and costs the cached-input price; the rest of
  the instruction, and the ticket, cost the ordinary input price. The answer costs the output price.
- `retrieval`: prompting plus `retrieved_tokens` more input tokens at the ordinary input price (retrieved
  passages differ every time, so they never hit the cache), plus embedding the ticket as the search query,
  at the embedding price.
- `finetune` and `distillation`: zero. Each small model runs on its own dedicated endpoint, paid by the hour,
  which is a *fixed* cost (exercise 3). Not zero because it is free: zero because the next request adds
  nothing, up to the endpoint's capacity.
- `mix`: `router_share` of requests go to the distilled small model and cost what distillation costs; the rest
  cost what prompting costs.

Use the helper `usd(tokens, usd_per_mtok)`. Any other strategy name is a `ValueError`.

<details><summary>💡 Hint 1: what to think about</summary>

Which tokens are billed at which of the three prices on the hosted model? Which of the five strategies
pay for the instruction at all? And which prices does retrieval add on top of the ones it inherits?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Compute prompting first and reuse it: retrieval adds two terms to it and mix blends it with distillation.
Split the instruction tokens into the cached share and the rest before pricing them.
</details>

In [ ]:
def request_cost_usd(strategy: str, scn: dict, snap: dict) -> float:
    """USD that one more request costs under `strategy`, not counting any one-off or monthly cost.

    Example (tickets of 200 tokens, answers of 50, no instruction, a model at 4 in / 8 out per Mtok):
        >>> request_cost_usd("prompting", scn, snap)
        0.0012

    Returns a float. Raises ValueError for a strategy name that is not in STRATEGIES.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _toy_snapshot() -> dict:
    return {"api_models": {"big": {"input_usd_per_mtok": 4.0, "output_usd_per_mtok": 8.0, "cache_read_usd_per_mtok": 1.0}},
            "open_models": {"teacher": {"input_usd_per_mtok": 2.0, "output_usd_per_mtok": 6.0}},
            "tinker": {"storage_usd_per_gb_month": 0.5, "models": {"small": {"train_usd_per_mtok": 3.0}}},
            "together": {"lora_sft": {"base": {"usd_per_mtok": 1.0, "minimum_charge_usd": 10.0}},
                         "dedicated_usd_per_gpu_hour": {"gpu": 2.0}},
            "fireworks": {"embedding_usd_per_mtok": 0.5}}


def _toy_scenario() -> dict:
    return {"months": 6, "input_tokens": 200, "output_tokens": 50, "instruction_tokens": 0, "cache_hit_share": 0.0,
            "retrieved_tokens": 0, "corpus_tokens": 0, "prompt_model": "big", "teacher_model": "teacher",
            "student_model": "small", "tuned_base": "base", "gpu": "gpu", "labelled_have": 0, "labelled_needed": 0,
            "label_usd_per_example": 0.0, "refresh_share": 0.0, "synthetic_examples": 0, "epochs": 1,
            "retrain_every_months": 0, "router_share": 0.8, "checkpoint_gb": 0.0, "endpoint_gpus": 1,
            "endpoint_hours_per_month": 0}


@_guard("request_cost_usd()")
def _check_request() -> None:
    snap, scn = _toy_snapshot(), _toy_scenario()
    # 200 tokens in at 4 and 50 out at 8 per million: 0.0008 + 0.0004
    assert abs(request_cost_usd("prompting", scn, snap) - 0.0012) < 1e-12, \
        "prompting with no instruction: the ticket at the input price plus the answer at the output price, each divided by a million"
    scn["instruction_tokens"], scn["cache_hit_share"] = 1000, 0.5
    # instruction: 500 cached at 1, 500 fresh at 4 -> 0.0005 + 0.002 on top of 0.0012
    assert abs(request_cost_usd("prompting", scn, snap) - 0.0037) < 1e-12, \
        "split the instruction: the cached share at the cached price, the remainder at the ordinary input price"
    scn["retrieved_tokens"] = 300
    assert abs(request_cost_usd("retrieval", scn, snap) - (0.0037 + 300 * 4 / MTOK + 200 * 0.5 / MTOK)) < 1e-12, \
        "retrieval = prompting + retrieved tokens at the ordinary input price + the ticket embedded at the embedding price"
    assert request_cost_usd("finetune", scn, snap) == 0, "finetune adds nothing per request: its cost is the endpoint, a fixed cost"
    assert request_cost_usd("distillation", scn, snap) == 0, \
        "distillation adds nothing per request either: its small model also runs on an always-on endpoint (exercise 3)"
    expected_mix = (1 - scn["router_share"]) * request_cost_usd("prompting", scn, snap)
    assert abs(request_cost_usd("mix", scn, snap) - expected_mix) < 1e-12, \
        "mix: router_share of requests cost what distillation costs (nothing per request); the rest, 1 - router_share, " \
        "cost what prompting costs"
    try:
        request_cost_usd("rag", scn, snap)
    except ValueError:
        pass
    else:
        raise AssertionError("an unknown strategy name should raise ValueError")
    print("exercise 2 looks right")


_try("exercise 2", _check_request)

In [ ]:
@_guard("the per-request table, which calls request_cost_usd()", 10)
def _show_requests() -> None:
    print(f"cost of ONE request, in USD, on the snapshot dated {SNAPSHOT['snapshot_date']}")
    base = request_cost_usd("prompting", SCENARIO, SNAPSHOT)
    for s in STRATEGIES:
        c = request_cost_usd(s, SCENARIO, SNAPSHOT)
        print(f"  {s:<13} {c:>12.6f}   {c / base:>6.2f} x prompting")
    c = request_cost_usd("prompting", SCENARIO, SNAPSHOT) * SCENARIO["requests_per_month"]
    print(f"\nprompting alone would cost USD {c:,.2f} a month at the scenario's volume of {SCENARIO['requests_per_month']:,} requests")
    print("a lower number per request is not a lower bill: the next exercise adds what you pay before and between requests")


_try("per-request table", _show_requests, needs=("exercise 2",))

## 4. Exercise 3: what you pay before, and between, requests

`fixed_costs_usd(strategy, scn, snap)` returns a list of `scn["months"]` numbers: the cost in USD that
lands in each month *regardless of volume*. Month 1 is index 0. The pieces:

- **Initial build**, in month 1. `retrieval`: embed `corpus_tokens` tokens. `finetune`: label the shortfall,
  `labelled_needed - labelled_have` examples (never negative) at `label_usd_per_example`, and train for
  `labelled_needed * (input_tokens + output_tokens) * epochs` tokens at Together's LoRA rate, but never
  below its **minimum charge** per job. `distillation` and `mix`: the teacher `scn["teacher_model"]`, priced
  in `snap["open_models"]`, labels `synthetic_examples` inputs (no cache: each call is the instruction plus the
  ticket in, the answer out, at the teacher's input and output prices), then the small model trains on
  `synthetic_examples * (input + output) * epochs` tokens at Tinker's train price. `prompting`: nothing.
- **Retraining**, when the data drifts: every `retrain_every_months` months after month 1, so indices
  `every`, `2*every`, and so on while below `months`; `0`, `None` or no such key means never. `retrieval` re-embeds
  `refresh_share` of the corpus. `finetune` relabels `refresh_share` of `labelled_needed` examples and trains again on
  the whole set. `distillation` and `mix` repeat their whole build.
- **Standing cost**, every month. `finetune`, `distillation` and `mix`: `endpoint_gpus * endpoint_hours_per_month`
  GPU-hours at the GPU's hourly price, for the small model's own always-on endpoint. `distillation` and `mix`
  also pay `checkpoint_gb` of storage at Tinker's monthly price.

Unknown strategy: `ValueError`.

<details><summary>💡 Hint 1: what to think about</summary>

Which strategies have a build at all, and which of those pay anything again when the data drifts? In which
month indices does a retrain land if it happens every `k` months, and what if `k` is larger than the year?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Work out three numbers per strategy: the build cost, the retrain cost and the monthly standing cost. Then
loop over the months once, adding the standing cost to every month, the build to the first and the retrain
to each retraining month. Take care of a job whose token cost falls under the minimum, and of already
having more labelled examples than you need.
</details>

In [ ]:
def fixed_costs_usd(strategy: str, scn: dict, snap: dict) -> list:
    """USD paid in each month of the year irrespective of volume, as a list of `scn["months"]` floats.

    Example (a retrieval corpus that costs 10 to embed, half of it re-embedded every 3 months, 6 months):
        >>> fixed_costs_usd("retrieval", scn, snap)
        [10.0, 0.0, 0.0, 5.0, 0.0, 0.0]

    Returns a list of floats. Raises ValueError for a strategy name that is not in STRATEGIES.
    """
    # YOUR CODE HERE
    raise NotImplementedError


@_guard("fixed_costs_usd()")
def _check_fixed() -> None:
    snap, scn = _toy_snapshot(), _toy_scenario()
    scn.update(corpus_tokens=20_000_000, refresh_share=0.5, retrain_every_months=3)
    got = fixed_costs_usd("retrieval", scn, snap)
    assert len(got) == scn["months"], "one number per month of the year"
    assert all(abs(a - b) < 1e-9 for a, b in zip(got, [10.0, 0, 0, 5.0, 0, 0])), \
        "retrieval: embed the whole corpus in month 1 (index 0), then re-embed refresh_share of it in each retraining month (index 3 here)"
    assert fixed_costs_usd("prompting", scn, snap) == [0.0] * 6, "prompting has no build, no retrain and no standing cost"
    scn.update(labelled_have=100, labelled_needed=400, label_usd_per_example=1.0, epochs=2,
               refresh_share=0.25, endpoint_gpus=2, endpoint_hours_per_month=10, retrain_every_months=0)
    # training: 400 * 250 tokens * 2 epochs = 200,000 tokens at 1.0 per Mtok = 0.2, below the minimum charge of 10
    got = fixed_costs_usd("finetune", scn, snap)
    assert abs(got[0] - (300 * 1.0 + 10.0 + 40.0)) < 1e-9, \
        "finetune build = label the shortfall (needed minus have) + max(training cost, minimum charge) + the first month's endpoint"
    assert all(abs(m - 40.0) < 1e-9 for m in got[1:]), "with no retraining every other month is just the endpoint: gpus x hours x the hourly price"
    assert fixed_costs_usd("finetune", dict(scn, retrain_every_months=None), snap) == got, "None also means never retrain"
    no_key = {k: v for k, v in scn.items() if k != "retrain_every_months"}
    assert fixed_costs_usd("finetune", no_key, snap) == got, "a scenario without retrain_every_months never retrains"
    scn.update(synthetic_examples=1000, instruction_tokens=300, checkpoint_gb=2.0)
    # the teacher labels 1000 inputs at 2 in and 6 out per Mtok; the small model trains at 3 per Mtok
    one_label = (300 + 200) * 2.0 / MTOK + 50 * 6.0 / MTOK
    build = 1000 * one_label + 1000 * 250 * 2 * 3.0 / MTOK
    got = fixed_costs_usd("distillation", scn, snap)
    assert abs(got[0] - (build + 1.0 + 40.0)) < 1e-9 and abs(got[1] - (1.0 + 40.0)) < 1e-9, \
        "distillation: the teacher labels at ITS prices (scn['teacher_model'] in snap['open_models']), the small model " \
        "trains, and storage and the small model's endpoint are paid every month"
    print("exercise 3 looks right")


_try("exercise 3", _check_fixed)

### A year, priced

A strategy's cost over the scenario's year is its fixed costs summed over the months, plus the months times the
monthly volume times the cost per request. `year_total_usd()` does that sum for you. It uses your two functions.
`endpoint_load()` says how many of a month's requests reach a small model's endpoint; later cells use it.

In [ ]:
def year_total_usd(strategy: str, scn: dict, snap: dict, volume: float) -> float:
    """Twelve-month (scn["months"]) cost in USD at `volume` requests per month. Given, not an exercise."""
    return sum(fixed_costs_usd(strategy, scn, snap)) + scn["months"] * volume * request_cost_usd(strategy, scn, snap)


def endpoint_load(strategy: str, scn: dict, volume: float) -> float:
    """Requests a month that reach a small model's endpoint at `volume` requests a month: all of them for finetune
    and distillation, the router's share for mix, none for the strategies that call a hosted model. Given."""
    return {"finetune": volume, "distillation": volume, "mix": scn["router_share"] * volume}.get(strategy, 0.0)


@_guard("the year table, which calls fixed_costs_usd()", 10)
def _show_year() -> None:
    vol = SCENARIO["requests_per_month"]
    rates = SNAPSHOT["fx"]["units_per_eur"]
    print(f"{SCENARIO['months']} months at {vol:,} requests a month; fixed = everything that does not depend on volume")
    print(f"{'strategy':<14}{'fixed, USD':>14}{'per request, USD':>19}" + "".join(f"{'total ' + c:>16}" for c in CURRENCIES))
    for s in STRATEGIES:
        fixed = sum(fixed_costs_usd(s, SCENARIO, SNAPSHOT))
        per = request_cost_usd(s, SCENARIO, SNAPSHOT)
        total = year_total_usd(s, SCENARIO, SNAPSHOT, vol)
        print(f"{s:<14}{fixed:>14,.2f}{per:>19.6f}" + "".join(f"{convert(total, 'USD', c, rates):>16,.0f}" for c in CURRENCIES))
    print(f"\nfinetune, distillation and mix each pay for an ASSUMED always-on endpoint: {SCENARIO['endpoint_gpus']} GPU for "
          f"{SCENARIO['endpoint_hours_per_month']} hours a month, from scenario.json. It is not a measurement of any deployment.")
    sched = fixed_costs_usd("distillation", SCENARIO, SNAPSHOT)
    steady = min(sched)
    extra = [str(i + 1) for i, x in enumerate(sched) if x > steady]
    print("\ndistillation fixed cost by month, USD:", [round(x, 2) for x in sched])
    print(f"every month pays at least {steady:,.2f}, the endpoint and the checkpoint storage; "
          + (f"month{'s' if len(extra) > 1 else ''} {', '.join(extra)} also pay{'' if len(extra) > 1 else 's'} "
             "for the teacher's labels and the training" if extra else "no month pays more"))


_try("a year, priced", _show_year, needs=("exercise 1", "exercise 2", "exercise 3"))

### How many requests one endpoint carries

An always-on endpoint costs the same busy or idle, but it is not bottomless: past some load every request waits
too long. Section 7 rules a small model out when more requests reach its endpoint than one endpoint carries, so
that number decides which strategies stay in the race at high volume. None of the sources this lesson cites
benchmarks its small model, so the capacity in `scenario.json` is **derived from a published benchmark of a
similar model, a proxy**, in steps the next cell prints:

1. **The benchmark.** NVIDIA's NIM documentation runs Llama-3.1-8B-Instruct on one H100 at several
   **concurrencies** (requests in flight at once) and gives, at each, the time to the first token (TTFT), the
   time between later tokens (ITL) and the output tokens a second of all requests together. Up to a point, more
   concurrency buys more tokens a second, at the price of a longer wait for each request. `serving_precision` in
   the scenario picks the run: the first in the snapshot at that precision, the one whose request is closest to
   this scenario's.
2. **The latency budget.** A request reaches the small model as the ticket and the answer (it was trained without
   the instruction, exercise 3) and takes about TTFT + (answer tokens - 1) x ITL from start to end. The cell scales
   TTFT by the ticket's length over the benchmark's prompt, and both times by the ratio of the two models' sizes,
   then walks up the table to the last concurrency at which a request is still within `latency_budget_ms`.
3. **Requests a second.** There, the benchmark's tokens a second, in and out together, are shared out among
   requests of the scenario's size and divided by the size ratio again. Counting an input token as dear as an
   output token overstates the work of a long prompt, so this step errs low.
4. **Requests a month.** An endpoint must carry the busiest hour, not the average one: the peak rate times the
   seconds the endpoint runs in a month, divided by `peak_to_average`.

Why the proxy is fair: both models are dense (every parameter works on every token) and of similar size, so their
work per token differs by about the ratio of their parameters; what the steps leave out is printed with the result.
The result is a **derived estimate from a proxy benchmark, not a measurement of this model**. `scenario.json` holds
it rounded down, and that is the number every later cell reads; section 7 shows what a tenth, or ten times, of it
would change. The same table gives the small models' latency that `scenario.json` holds: one request on an idle
endpoint, rounded up, the same for both small models (one model, one GPU), and for the mix the average over its
traffic. Under load a request takes longer; the capacity is the load at which it still fits the budget.

In [ ]:
def endpoint_capacity(scn: dict, snap: dict, run: dict) -> dict:
    """Requests a month that one small-model endpoint carries within the latency budget, derived from one `run` of
    the snapshot's proxy benchmark, with the number behind every step so that the steps can be printed. Given."""
    bench = snap["serving_benchmark"]
    size = snap["tinker"]["models"][scn["student_model"]]["params"] / bench["model_params"]   # served model / proxy
    tokens_in, tokens_out = scn["input_tokens"], scn["output_tokens"]   # the ticket and the answer, no instruction
    rows = [dict(row, request_ms=size * (row["ttft_ms"] * tokens_in / run["input_tokens"]
                                         + (tokens_out - 1) * row["itl_ms"]))
            for row in run["rows"]]
    at = None
    for row in rows:   # up the table, in order of concurrency, until a request would take longer than the budget
        if row["request_ms"] > scn["latency_budget_ms"]:
            break
        at = row
    out = {"size": size, "rows": rows, "at": at, "tokens_per_s": 0.0, "per_s": 0.0, "per_month": 0.0}
    if at is not None:
        out["tokens_per_s"] = at["output_tokens_per_s"] * (run["input_tokens"] + run["output_tokens"]) / run["output_tokens"]
        out["per_s"] = out["tokens_per_s"] / (tokens_in + tokens_out) / size
        out["per_month"] = out["per_s"] * 3600 * scn["endpoint_hours_per_month"] / scn["peak_to_average"]
    return out


def _show_capacity() -> None:
    bench, student = SNAPSHOT["serving_benchmark"], SCENARIO["student_model"]
    runs = [r for r in bench["runs"] if r["precision"] == SCENARIO["serving_precision"]]
    if not runs:
        print(f"the snapshot has no benchmark run at {SCENARIO['serving_precision']}: set serving_precision in "
              f"scenario.json to one of {sorted({r['precision'] for r in bench['runs']})}")
        return
    run = runs[0]
    cap = endpoint_capacity(SCENARIO, SNAPSHOT, run)
    small = SNAPSHOT["tinker"]["models"][student]
    tin, tout, budget = SCENARIO["input_tokens"], SCENARIO["output_tokens"], SCENARIO["latency_budget_ms"]
    print(f"how many requests one small-model endpoint carries: a DERIVED ESTIMATE from a proxy benchmark, "
          f"not a measurement of {student}\n")
    print(f"the benchmark, read {bench['read_on']}: {bench['model']} on {bench['gpu']}, {bench['software']}, "
          f"{run['precision']} weights, {run['input_tokens']} tokens in and {run['output_tokens']} out per request")
    print(f"the proxy: {bench['model']} is {bench['model_arch'].lower()} with {bench['model_params'] / 1e9:g}B parameters "
          f"and {student} {small['arch'].lower()} with {small['params'] / 1e9:g}B, as their publishers state them; assuming "
          f"work per token in proportion to size, times are multiplied by {cap['size']:.3f} and rates divided by it")
    print(f"a request reaches the small model as the ticket and the answer, {tin} tokens in and {tout} out, without the "
          f"{SCENARIO['instruction_tokens']:,}-token instruction (an assumption: the small models are trained without it, "
          f"exercise 3); the latency budget is {budget} ms\n")
    print(f"  {'concurrency':>11}{'TTFT':>12}{'ITL':>10}{'output tokens/s':>17}{'one request here':>19}")
    for row in cap["rows"]:
        print(f"  {row['concurrency']:>11}{row['ttft_ms']:>9.2f} ms{row['itl_ms']:>7.2f} ms"
              f"{row['output_tokens_per_s']:>17,.2f}{row['request_ms']:>16,.1f} ms   "
              + ("within the budget" if row["request_ms"] <= budget else "over it"))
    print(f"  (one request here = {cap['size']:.3f} x (TTFT x {tin} / {run['input_tokens']} + ({tout} - 1) x ITL), "
          "assuming the time to the first token grows in proportion to the prompt)\n")
    at = cap["at"]
    if at is None:
        print(f"no concurrency in this run keeps a request within {budget} ms: within that budget one endpoint carries none")
        return
    beyond = next((r for r in cap["rows"] if r["concurrency"] > at["concurrency"]), None)
    print(f"1. the operating point: the last concurrency before a request passes the budget is {at['concurrency']}"
          + (f" (the table's next, {beyond['concurrency']}, is over it: the true limit lies between them, so this errs low)"
             if beyond is not None else " (the last in the table)"))
    print(f"2. requests a second there: {at['output_tokens_per_s']:,.2f} output tokens/s x ({run['input_tokens']} + "
          f"{run['output_tokens']}) / {run['output_tokens']} = {cap['tokens_per_s']:,.2f} tokens/s in and out, shared "
          f"among requests of {tin} + {tout} tokens and divided by {cap['size']:.3f}: {cap['per_s']:.2f} requests a second "
          "at the peak (counting an input token as dear as an output token: an assumption that errs low)")
    print(f"3. requests a month: that rate x 3,600 seconds x {SCENARIO['endpoint_hours_per_month']} hours, divided by "
          f"peak_to_average {SCENARIO['peak_to_average']}: {cap['per_month']:,.0f}")
    held, rounded = SCENARIO["endpoint_capacity_requests_per_month"], cap["per_month"] // 100_000 * 100_000
    print(f"scenario.json holds {held:,}; the estimate rounded down to a multiple of 100,000 is {rounded:,.0f}: "
          + ("the same" if held == rounded else "they differ, and every later cell reads the value in scenario.json"))
    week, busy = 7 * 24, 40
    print(f"\npeak_to_average is an assumption: tickets spread evenly over {busy} working hours of a week's {week}, and "
          f"none outside them, would peak at {week / busy:.1f} times their average")
    peak = max(cap["rows"], key=lambda r: r["output_tokens_per_s"])
    if peak["request_ms"] > budget:
        print(f"the most output tokens a second in this run, {peak['output_tokens_per_s']:,.2f} at concurrency "
              f"{peak['concurrency']}, would put one request at {peak['request_ms']:,.1f} ms: the highest throughput "
              "a GPU reaches is not what it carries within a latency budget")
    alone, lat, share = cap["rows"][0], SCENARIO["assumed_latency_ms"], SCENARIO["router_share"]
    idle = -(-alone["request_ms"] // 10) * 10                       # rounded up to a multiple of 10 ms
    mix_ms = round(share * idle + (1 - share) * lat["prompting"])   # to the whole millisecond before comparing
    print(f"\nlatency: one request on an idle endpoint (concurrency {alone['concurrency']}) takes "
          f"{alone['request_ms']:,.1f} ms, {idle:,.0f} rounded up to a multiple of 10, for finetune and distillation alike "
          f"(one small model, one GPU);\nmix sends {share:.0%} of its requests there and the rest to the hosted model, "
          f"assumed at {lat['prompting']} ms: {mix_ms:,} ms on average")
    held = (lat["finetune"], lat["distillation"], lat["mix"])
    print(f"scenario.json holds {held[0]}, {held[1]} and {held[2]} ms for finetune, distillation and mix: "
          + ("the same" if held == (idle, idle, mix_ms) else "they differ, and section 7 reads the values in scenario.json"))
    busy = share * at["request_ms"] + (1 - share) * lat["prompting"]
    print(f"under load a request takes longer: {at['request_ms']:,.1f} ms at the operating point, and mix's average "
          f"{busy:,.1f} ms there; " + ("both are within" if max(at["request_ms"], busy) <= budget else "mix's is over")
          + f" the {budget} ms budget")
    for what, ms in (("a request at the operating point", at["request_ms"]), ("mix's average there", busy)):
        if 0 <= budget - ms <= 0.01 * budget:   # a decision this close to its constraint turns on the estimate
            print(f"CLOSE CALL: {what} is {ms:,.1f} ms, inside the {budget} ms budget by only {budget - ms:,.1f} ms "
                  f"({(budget - ms) / budget:.1%} of the budget): a small change in an assumption would flip the statement above about the budget")
    per_size = at["request_ms"] / cap["size"]   # a request here at a size ratio of 1
    top = budget / per_size                       # the largest size ratio at which this row still meets the budget
    low = budget / (beyond["request_ms"] / cap["size"]) if beyond is not None else None   # above this the next row would fit
    mix_top = (budget - (1 - share) * lat["prompting"]) / share / per_size   # the largest ratio at which mix's average still fits
    over = json.loads(json.dumps(SNAPSHOT))
    over["tinker"]["models"][student]["params"] = bench["model_params"] * top * 1.001
    past = endpoint_capacity(SCENARIO, over, run)
    print(f"sensitivity to the size ratio (served model's parameters over the proxy's; this lesson uses {cap['size']:.3f}): the operating "
          f"point stays at concurrency {at['concurrency']} for ratios " + (f"above {low:.3f} and " if low else "") + f"up to {top:.3f}; "
          f"mix's average stays within the budget only up to {mix_top:.3f}. A served model counted larger (for example with an "
          f"encoder or other weights that do no work on text) can pass them: just past {top:.3f} the operating point falls to "
          + (f"concurrency {past['at']['concurrency']} and the capacity to {past['per_month']:,.0f} a month"
             if past["at"] else "none: no concurrency keeps a request within the budget")
          + f", against {cap['per_month']:,.0f} here")
    print("not modelled: the served model's linear attention in three layers of four, the cost of serving a LoRA "
          "adapter, the host's own serving software and bursts within the busiest hour; the precision the host serves "
          f"at is an assumption, serving_precision {SCENARIO['serving_precision']}")
    print("\nthe same steps on every run in the snapshot:")
    months = []
    for r in bench["runs"]:
        c = endpoint_capacity(SCENARIO, SNAPSHOT, r)
        months.append(c["per_month"])
        where = f"concurrency {c['at']['concurrency']}" if c["at"] else "no concurrency within the budget"
        print(f"  {r['precision']:<5}{r['input_tokens']:>5} tokens in, {r['output_tokens']:>5} out: "
              f"{c['per_month']:>12,.0f} a month, at {where}")
    if min(months) > 0 and max(months) > 2 * min(months):
        print(f"the largest is {max(months) / min(months):.1f} times the smallest: which run stands in for the endpoint, "
              "its precision and the shape of its requests, is itself an assumption that moves the estimate")


_show_capacity()

## 5. Exercise 4: where one strategy overtakes another

Over `months` months at a constant monthly volume `v`, a strategy costs `fixed + months * v * per_request`.
That is a straight line in `v`. Two strategies **cross** where their lines meet.

`breakeven_volume(fixed_a, per_a, fixed_b, per_b, months)` answers: *above what monthly volume is B strictly
cheaper than A?* Return that volume as a float. There are three kinds of answer:

- B costs less per request than A and **more** up front: they meet at a positive volume; return it.
- B is cheaper at **every** volume, however small: return `0.0`. That includes B costing less per request and
  no more up front, and B costing the same per request but less up front.
- There is no volume above which B is cheaper: return `None`. That includes B costing more per request, and
  B costing the same per request but not less up front.

Compare the numbers exactly, with no tolerance: a millionth of a dollar more is still more.

<details><summary>💡 Hint 1: what to think about</summary>

Sketch the two lines. Which one is higher at zero volume, and which one rises faster? A line that starts
higher and rises faster can never catch one that starts lower and rises slower.
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Decide the case from the per-request costs first: B's slope below A's, equal, or above. Only when B's slope
is lower is there a crossing to compute: set the two totals equal and solve for the volume. Compare
the up-front costs to see whether the crossing is in the future (positive) or already behind you.
</details>

In [ ]:
def breakeven_volume(fixed_a: float, per_a: float, fixed_b: float, per_b: float, months: int) -> "float | None":
    """The monthly volume above which strategy B is strictly cheaper over `months` months than A.

    Example (A costs 0 up front and 1 a request, B costs 120 up front and 0.5 a request, 12 months):
        >>> breakeven_volume(0, 1, 120, 0.5, 12)
        20.0

    Returns a float, `0.0` when B is cheaper at every volume, or None when there is no such volume.
    """
    # YOUR CODE HERE
    raise NotImplementedError


@_guard("breakeven_volume()")
def _check_breakeven() -> None:
    assert abs(breakeven_volume(0, 1, 120, 0.5, 12) - 20.0) < 1e-9, \
        "120 extra up front, 0.5 saved per request, over 12 months: 120 / (12 x 0.5)"
    assert abs(breakeven_volume(100, 3, 400, 1, 6) - 25.0) < 1e-9, \
        "the saving per request is per_a - per_b, and it accrues over `months` months"
    assert breakeven_volume(0, 1, 0, 2, 12) is None, "B costs more per request and not less up front: it is never cheaper"
    assert breakeven_volume(0, 1, 50, 2, 12) is None, "B is dearer on both counts: never"
    assert breakeven_volume(100, 1, 50, 2, 12) is None, "cheaper up front but dearer per request is not 'cheaper above some volume'"
    assert breakeven_volume(100, 1, 50, 0.5, 12) == 0.0, "cheaper on both counts: cheaper at every volume, so 0.0, not a negative number"
    assert breakeven_volume(100, 1, 100, 0.5, 12) == 0.0, "same up front, cheaper per request: cheaper from the first request"
    assert breakeven_volume(100, 1, 40, 1, 12) == 0.0, "same per request, cheaper up front: cheaper at every volume"
    assert breakeven_volume(100, 1, 100, 1, 12) is None, "identical lines: B is never strictly cheaper"
    print("exercise 4 looks right")


_try("exercise 4", _check_breakeven)

In [ ]:
@_guard("the break-even table, which calls your functions", 10)
def _show_breakevens() -> None:
    vol = SCENARIO["requests_per_month"]
    print(f"break-even volumes, requests a month, over {SCENARIO['months']} months (cost only: quality and latency come later)\n")
    print(f"{'is B cheaper than A above ...':<46}{'break-even':>14}   where the scenario sits")
    opts = {s: (sum(fixed_costs_usd(s, SCENARIO, SNAPSHOT)), request_cost_usd(s, SCENARIO, SNAPSHOT)) for s in STRATEGIES}
    for a, b in (("prompting", "distillation"), ("prompting", "finetune"), ("finetune", "distillation"),
                 ("prompting", "retrieval"), ("distillation", "mix")):
        v = breakeven_volume(*opts[a], *opts[b], SCENARIO["months"])
        label = f"A = {a}, B = {b}"
        if v is None:
            (fa, pa), (fb, pb) = opts[a], opts[b]
            why = ("B costs the same per request and no less up front" if pb == pa
                   else "B costs less up front but more per request: it is cheaper only BELOW some volume" if fb < fa
                   else "B costs no less up front and more per request: dominated")
            print(f"{label:<46}{'never':>14}   {why}")
        elif v == 0:
            print(f"{label:<46}{v:>14,.0f}   B is cheaper at every volume")
        else:
            where = "below" if vol < v else "above" if vol > v else "exactly at"
            print(f"{label:<46}{v:>14,.0f}   the scenario's {vol:,} is {where} it")
    print("\n'never' means there is no volume above which B is cheaper than A; a break-even of 0 means B is cheaper at")
    print("every volume. All of this assumes the same volume every month of the year.")


_try("break-even table", _show_breakevens, needs=("exercise 2", "exercise 3", "exercise 4"))

## 6. Exercise 5: the cheapest strategy at every volume

Pairs are not a decision. With five strategies you want the **ladder**: which strategy is cheapest as the
volume grows from nothing, and the volume at which each rung takes over.

`ladder(options, months)` takes `options`, a dict `{name: (fixed_total, per_request)}`, and returns a list of
`(name, from_volume)` tuples in order of volume, starting at `0.0`. Rules:

- The first entry is the cheapest at volume zero: the lowest `fixed_total`, and among equals the lowest
  `per_request`.
- Each next entry is the strategy that becomes cheapest soonest after the current one: among options with a
  **lower** `per_request`, the one whose crossing with the current strategy (exercise 4) comes first. If two
  cross at the same volume, take the one with the lower `per_request`; the other is never the cheapest.
- A strategy that is never the cheapest does not appear. An option identical to an earlier one in
  `(fixed_total, per_request)` also does not appear: the first in dict order stands for both.

Stop when no remaining option has a lower per-request cost. The loop must end: the current strategy's
`per_request` falls every step.

<details><summary>💡 Hint 1: what to think about</summary>

Start from the strategy with the least to pay up front. What can overtake it, and what can never overtake it
however large the volume? Once you stand on a rung, which strategies are still in the race?
</details>

<details><summary>💡 Hint 2: the approach, in words</summary>

Pick the starting rung. Then repeat: look at every option whose per-request cost is strictly lower than the
current one's, compute where it crosses the current one, and move to the nearest crossing (the cheaper per
request on a tie). Record the volume you moved at. When nothing is left that is cheaper per request, stop.
</details>

In [ ]:
def ladder(options: dict, months: int) -> list:
    """The strategies that are cheapest somewhere, with the monthly volume from which each is.

    Example (A is cheap to start, B costs 30 more up front and 2 less per request, 1 month):
        >>> ladder({"A": (0, 4), "B": (30, 2)}, 1)
        [('A', 0.0), ('B', 15.0)]

    Returns a list of (name, from_volume), the first at 0.0. A dominated strategy never appears.
    """
    # YOUR CODE HERE
    raise NotImplementedError


@_guard("ladder()")
def _check_ladder() -> None:
    got = ladder({"A": (0, 10), "B": (100, 5), "C": (150, 1)}, 1)
    assert [n for n, _ in got] == ["A", "C"], \
        "B is cheaper than A only above a volume where C is already cheaper than B: B is never the cheapest, so it must not appear"
    assert abs(got[1][1] - 150 / 9) < 1e-9, "C overtakes A where 150 + v = 10 v"
    got = ladder({"slow": (0, 3), "fast": (120, 1)}, 12)
    assert got[0] == ("slow", 0.0) and got[1][0] == "fast" and abs(got[1][1] - 5.0) < 1e-9, \
        "the first rung starts at 0.0; the next starts at the crossing volume"
    got = ladder({"big": (500, 0.5), "cheap": (10, 1), "mid": (100, 0.8)}, 1)
    assert [n for n, _ in got] == ["cheap", "mid", "big"], \
        "order the rungs by volume, not by name or by position in the dict"
    got = ladder({"dear": (50, 2), "cheap": (10, 1)}, 12)
    assert got == [("cheap", 0.0)], "a strategy dearer on both counts is dominated: the ladder has one rung"
    print("exercise 5 looks right")


_try("exercise 5", _check_ladder)

In [ ]:
@_guard("the ladder, which calls your functions", 10)
def _show_ladder() -> None:
    opts = {s: (sum(fixed_costs_usd(s, SCENARIO, SNAPSHOT)), request_cost_usd(s, SCENARIO, SNAPSHOT)) for s in STRATEGIES}
    with _time_limit(2, "ladder()"):
        rungs = ladder(opts, SCENARIO["months"])
    print(f"cheapest strategy by monthly volume, cost only, {SCENARIO['months']} months, snapshot dated {SNAPSHOT['snapshot_date']}\n")
    for i, (name, start) in enumerate(rungs):
        end = rungs[i + 1][1] if i + 1 < len(rungs) else None
        span = f"from {start:>12,.0f} requests a month" + (f" up to {end:,.0f}" if end is not None else " and above")
        print(f"  {name:<13}{span}")
    never = [s for s in STRATEGIES if s not in {n for n, _ in rungs}]
    print("\nnever the cheapest at any volume:", ", ".join(never) if never else "none")
    cap = SCENARIO["endpoint_capacity_requests_per_month"]
    for name, start in rungs:
        share = endpoint_load(name, SCENARIO, 1.0)   # the share of each request that reaches a small model's endpoint
        if share:
            limit = cap / share
            print(f"{name} starts winning at {start:,.0f} a month; its one endpoint is estimated (section 4) to carry up to "
                  f"{limit:,.0f} " + ("a month, and above that more endpoints would be needed: the ladder prices only one"
                                      if start <= limit else "a month, so that rung starts beyond what the endpoint can carry"))


_try("the ladder", _show_ladder, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 7. From cheapest to chosen: quality, latency and data

Cheapest is not chosen. A strategy is **feasible** only if it meets the quality target, fits the latency
budget, has the data it needs and can carry the volume. `feasible()` below returns the reasons a strategy
fails, so an empty list means it passes. It is given: read it, then change a number in the scenario and
run the cell again.

Quality and latency come from the scenario's **assumed** tables. A strategy's latency is that of its average
request; the small models' latency, and so the mix's, is the estimate from section 4. In a real project you
would measure them on your own evaluation set; here they are stated so that you can see how the decision
responds. What one endpoint carries is also the estimate from section 4, so the table at the end of the cell
makes the choice again with a tenth of it and with ten times it.

In [ ]:
def feasible(strategy: str, scn: dict, volume: float) -> list:
    """Reasons `strategy` is ruled out at `volume` requests a month; an empty list means it is feasible. Given."""
    why = []
    if scn["assumed_quality"][strategy] < scn["quality_target"]:
        why.append(f"quality {scn['assumed_quality'][strategy]} is below the target {scn['quality_target']}")
    if scn["assumed_latency_ms"][strategy] > scn["latency_budget_ms"]:
        why.append(f"latency {scn['assumed_latency_ms'][strategy]} ms is over the budget {scn['latency_budget_ms']} ms")
    if strategy in ("distillation", "mix") and scn["unlabelled_inputs"] < scn["synthetic_examples"]:
        why.append(f"only {scn['unlabelled_inputs']:,} unlabelled inputs for {scn['synthetic_examples']:,} teacher labels")
    load = endpoint_load(strategy, scn, volume)
    if load > scn["endpoint_capacity_requests_per_month"]:
        why.append(f"{load:,.0f} requests a month reach its endpoint, over one endpoint's estimated capacity")
    return why


def decide(scn: dict, snap: dict, volume: float) -> list:
    """Every strategy with its year total and reasons, cheapest first; feasible strategies sort ahead. Given."""
    rows = [(s, year_total_usd(s, scn, snap, volume), feasible(s, scn, volume)) for s in STRATEGIES]
    return sorted(rows, key=lambda r: (bool(r[2]), r[1]))   # sorted is stable: ties keep STRATEGIES order


@_guard("the decision, which calls fixed_costs_usd()", 10)
def _show_decision() -> None:
    vol = SCENARIO["requests_per_month"]
    rows = decide(SCENARIO, SNAPSHOT, vol)
    print(f"decision at {vol:,} requests a month, quality target {SCENARIO['quality_target']}, latency budget {SCENARIO['latency_budget_ms']} ms\n")
    for s, total, why in rows:
        print(f"  {s:<13}{total:>14,.0f} USD   {'FEASIBLE' if not why else 'ruled out: ' + '; '.join(why)}")
    winner = rows[0][0] if not rows[0][2] else None
    print(f"\nchosen: {winner}" if winner else "\nnothing is feasible: relax a constraint or change the data")
    print(f"(on the snapshot dated {SNAPSHOT['snapshot_date']} and this imaginary scenario's assumptions: a worked example, not a recommendation of any provider)")
    if winner in ("distillation", "mix"):
        print(f"{winner} trains the small model on the open teacher's answers: it is an option only if the teacher's licence "
              "and the host's terms permit that (section 1)")
    tight = dict(SCENARIO, quality_target=0.93)
    t_rows = decide(tight, SNAPSHOT, vol)
    t_win = t_rows[0][0] if not t_rows[0][2] else None
    print(f"with the quality target raised to {tight['quality_target']}: chosen {t_win}")
    relaxed = dict(SCENARIO, quality_target=0.85, latency_budget_ms=1000)
    cap = SCENARIO["endpoint_capacity_requests_per_month"]
    columns = {"scenario": SCENARIO, "relaxed": relaxed,
               "capacity / 10": dict(SCENARIO, endpoint_capacity_requests_per_month=cap / 10),
               "capacity x 10": dict(SCENARIO, endpoint_capacity_requests_per_month=cap * 10)}
    print(f"\nthe choice by monthly volume: under the scenario's constraints; under a relaxed pair (quality "
          f"{relaxed['quality_target']}, latency {relaxed['latency_budget_ms']} ms);\nand under the scenario's constraints "
          f"with one endpoint carrying a tenth, or ten times, the {cap:,} requests a month estimated in section 4:")
    print(f"  {'requests a month':>18}" + "".join(f"{name:>18}" for name in columns))
    volumes = (vol // 100, vol // 40, vol // 4, vol, vol * 8, vol * 40)
    differs = dict.fromkeys(columns, 0)
    for v in volumes:
        picks = {}
        for name, scn in columns.items():
            r = decide(scn, SNAPSHOT, v)
            picks[name] = r[0][0] if not r[0][2] else "nothing feasible"
            differs[name] += picks[name] != picks["scenario"]
        print(f"  {v:>18,}" + "".join(f"{pick:>18}" for pick in picks.values()))
    print(f"the relaxed pair changes the choice at {differs['relaxed']} of {len(volumes)} volumes; a tenth of the capacity "
          f"at {differs['capacity / 10']}, ten times the capacity at {differs['capacity x 10']}")
    if differs["capacity / 10"] or differs["capacity x 10"]:
        print("where a capacity column differs from the scenario's, the choice turns on the endpoint's capacity, an "
              "estimate from a proxy benchmark: measure your own model before you rely on it")


_try("the decision", _show_decision, needs=("exercise 2", "exercise 3"))

## 8. Sensitivity: what if prices move?

Prices change every quarter, so a calculator that gives one number is a liability. Below, the snapshot is
**re-priced** and the decision recomputed. `scaled()` multiplies one group of prices at a time: the hosted
model's (`api`), the open teacher's (`teacher`), the training prices (`tuning`: Tinker's and Together's), and
the dedicated GPU (`gpu`). A minimum charge is not a price per token, so it stays.

In [ ]:
def scaled(snap: dict, api: float = 1.0, teacher: float = 1.0, tuning: float = 1.0, gpu: float = 1.0) -> dict:
    """A copy of `snap` with the named price groups multiplied. Given."""
    out = copy.deepcopy(snap)
    for m in out["api_models"].values():
        for k in ("input_usd_per_mtok", "output_usd_per_mtok", "cache_read_usd_per_mtok"):
            m[k] *= api
    for m in out["open_models"].values():
        for k in ("input_usd_per_mtok", "output_usd_per_mtok"):
            m[k] *= teacher
    for m in out["tinker"]["models"].values():
        m["train_usd_per_mtok"] *= tuning
    for m in out["together"]["lora_sft"].values():
        m["usd_per_mtok"] *= tuning
    for k in out["together"]["dedicated_usd_per_gpu_hour"]:
        out["together"]["dedicated_usd_per_gpu_hour"][k] *= gpu
    return out


MOVES = [("as in the snapshot", {}), ("hosted model 50% cheaper", {"api": 0.5}), ("hosted model twice the price", {"api": 2.0}),
         ("open teacher twice the price", {"teacher": 2.0}), ("training twice the price", {"tuning": 2.0}),
         ("dedicated GPU half the price", {"gpu": 0.5}), ("dedicated GPU twice the price", {"gpu": 2.0})]


@_guard("the sensitivity table, which calls your functions", 10)
def _show_sensitivity() -> None:
    vol = SCENARIO["requests_per_month"]
    print(f"{'price move':<36}{'chosen':>14}{'its year, USD':>16}{'first break-even':>20}{'last break-even':>18}")
    chosen_names, two_rungs, beyond = [], [], []
    cap = SCENARIO["endpoint_capacity_requests_per_month"]
    for label, kw in MOVES:
        snap = scaled(SNAPSHOT, **kw)
        rows = decide(SCENARIO, snap, vol)
        best = rows[0] if not rows[0][2] else None
        opts = {s: (sum(fixed_costs_usd(s, SCENARIO, snap)), request_cost_usd(s, SCENARIO, snap)) for s in STRATEGIES}
        with _time_limit(2, "ladder()"):
            rungs = ladder(opts, SCENARIO["months"])
        if any(endpoint_load(n, SCENARIO, v) > cap for n, v in rungs[1:]):
            beyond.append(label)   # the same rule as feasible(): more requests than one endpoint carries
        first = f"{rungs[1][1]:,.0f}" if len(rungs) > 1 else "none"
        last = f"{rungs[-1][1]:,.0f}" if len(rungs) > 1 else "none"
        chosen_names.append(best[0] if best else None)
        two_rungs.append(len(rungs) == 2)
        print(f"{label:<36}{(best[0] if best else '-'):>14}{(best[1] if best else float('nan')):>16,.0f}{first:>20}{last:>18}")
    changed = sum(n != chosen_names[0] for n in chosen_names)
    print(f"\nthe chosen strategy differs from the snapshot's in {changed} of {len(MOVES) - 1} moves;")
    print("the break-even columns are the volumes (requests a month) at which the second and the last rung of the ladder take over;")
    print("they show how far those volumes move" + (" even though the choice at the scenario's volume does not." if changed == 0
          else ", alongside the choice itself."))
    if all(two_rungs):
        print("in every move the second rung of the ladder is also its last, so the two break-even columns agree")
    if beyond:
        print(f"in {len(beyond)} of {len(MOVES)} rows ({'; '.join(beyond)}) a rung starts where its endpoint would get more "
              f"than the {cap:,} requests a month one endpoint is estimated to carry: there the ladder, which prices one "
              f"endpoint, leaves out the extra endpoints that volume would need")


_try("sensitivity", _show_sensitivity, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 9. Keeping the snapshot honest

This notebook never touches the network: its prices are in a file with a date. A real calculator refreshes
that file. Tinker publishes its price table as machine-readable `models.json` and `serverless.json` beside
its documentation, and says plainly not to scrape the HTML (`claims.yaml`). Prices there are strings with a
leading dollar sign, so the first job of an ingest is to turn a string into a number. `parse_price()` does that.
The cell parses the small model's row as `models.json` held it when the snapshot's Tinker block was read (copied
by hand, so this checks the transcription, not today's price) and compares the one price the snapshot keeps
from it, the train price: the lesson serves the small model elsewhere (section 1).

A refresh script would fetch the file, convert each row, write it to `price_snapshot.json` with today's date,
and re-run this notebook's checks. It is deliberately not part of this notebook: a lesson that fetches is a
lesson that gives different answers on different days.

In [ ]:
def parse_price(text: str) -> float:
    """'$1.463' -> 1.463: the string form Tinker's models.json uses for a price per million tokens. Given."""
    return float(text.strip().lstrip("$").replace(",", ""))


_SAMPLE_ROW = '{"tinker_id": "Qwen/Qwen3.5-9B", "prefill": "$0.30", "sample": "$0.807", "train": "$1.463"}'
_row = json.loads(_SAMPLE_ROW)
_snap_row = SNAPSHOT["tinker"]["models"][_row["tinker_id"]]
_agree = parse_price(_row["train"]) == _snap_row["train_usd_per_mtok"]
print("sample row parsed:", {k: parse_price(v) for k, v in _row.items() if k != "tinker_id"})
print(f"the snapshot's train price agrees with the sample row: {_agree} (Tinker block read on {SNAPSHOT['tinker']['read_on']})")
_dates = [SNAPSHOT["fx"]["rate_date"], SNAPSHOT["serving_a_fine_tune"]["read_on"]]
_dates += [row["read_on"] for block in ("api_models", "open_models") for row in SNAPSHOT[block].values()]
_dates += [SNAPSHOT[block]["read_on"] for block in ("tinker", "together", "fireworks", "serving_benchmark")]
print(f"a price older than a quarter is a reason to refresh: this snapshot is dated {SNAPSHOT['snapshot_date']}, "
      f"and its oldest figure dates from {min(_dates)}")

## 10. Common mistakes

- **Pricing a fine-tuned model per token without asking how it is served.** A dedicated endpoint costs the same
  whether it serves one request or a million, so its per-request price is a lie at low volume.
- **Forgetting retraining.** A model trained once is priced for a year of data that changes. Drift turns a
  one-off cost into a recurring one.
- **Ignoring the minimum charge.** A small training job costs the minimum, not tokens times rate.
- **Dividing where you should multiply when converting currencies.** The ECB quotes units per euro.
- **Picking the cheapest strategy without the constraints.** The cheapest line on the ladder may miss the
  quality target, the latency budget or the data you have.
- **Trusting an assumed quality figure as a measurement.** Here they are assumptions on purpose; on your task, measure them.
- **Taking a GPU's highest throughput as its capacity.** The most tokens a second come at a concurrency where every
  request waits longer; what one GPU carries is its rate within your latency budget, at your traffic's busiest
  hour, and a benchmark of another model is an estimate, not a measurement of yours. The next cell puts numbers
  on it.

In [ ]:
def _show_peak_mistake() -> None:
    runs = [r for r in SNAPSHOT["serving_benchmark"]["runs"] if r["precision"] == SCENARIO["serving_precision"]]
    if not runs:
        print(f"the snapshot has no benchmark run at {SCENARIO['serving_precision']}: see section 4")
        return
    budget = SCENARIO["latency_budget_ms"]
    within = endpoint_capacity(SCENARIO, SNAPSHOT, runs[0])
    peak = max(within["rows"], key=lambda r: r["output_tokens_per_s"])
    at_peak = endpoint_capacity(dict(SCENARIO, latency_budget_ms=peak["request_ms"]), SNAPSHOT, runs[0])
    if at_peak["at"] is None:
        print("the run's highest throughput comes after a concurrency whose requests take longer: see section 4")
        return
    print(f"read at the run's highest throughput, concurrency {at_peak['at']['concurrency']}: "
          f"{at_peak['per_month']:,.0f} requests a month, each taking {at_peak['at']['request_ms']:,.1f} ms")
    if within["at"] is None:
        print(f"read within the {budget} ms budget: no concurrency in the run keeps a request within it")
        return
    print(f"read within the {budget} ms budget, concurrency {within['at']['concurrency']}: {within['per_month']:,.0f} "
          f"requests a month, so the first figure is {at_peak['per_month'] / within['per_month']:.2f} times this one"
          + (f", and every request it counts misses the {budget} ms budget" if at_peak["at"]["request_ms"] > budget else ""))


_show_peak_mistake()

## 11. Self-check

1. A fine-tuned model on a dedicated endpoint is paid by the hour. What does that do to its place on the ladder?
   - (a) Nothing changes, because every strategy on the ladder is priced per token and per request.
   - (b) It lowers the strategy's cost per request at every volume, so it wins earlier and earlier.
   - (c) It turns a per-request cost into a fixed one, so the strategy can win only above some volume.
   - (d) It rules the strategy out, because an always-on machine cannot be compared with the others.
2. You halve the price of the hosted large model and the ladder's first break-even moves. In which direction, and why?
   - (a) Down, because staying on the hosted model has become so cheap that a replacement is needed sooner.
   - (b) Up, because a replacement now saves less on each request, so it needs more volume to repay its fixed cost.
   - (c) It does not move, because a break-even depends only on the fixed costs of the strategies compared.
   - (d) Down, because the fixed costs of every replacement fall by the same half as the hosted price.
3. A strategy appears nowhere on the ladder. Which statement must be true?
   - (a) At every monthly volume, some other strategy costs no more over the year than it does.
   - (b) The strategy fails the quality target or the latency budget at every volume.
   - (c) Its break-even against the first rung is exactly zero, so the ladder leaves it out.
   - (d) The ladder has a bug, because every strategy should be the cheapest somewhere.
4. Which part of this calculator should survive a quarter of price changes unchanged?
   - (a) The printed year totals, because they were computed from the snapshot by the code.
   - (b) The snapshot, because the providers publish their prices only once.
   - (c) The structure: fixed against variable cost, retraining, the ladder and the feasibility test.
   - (d) The exchange rates, because currencies are quoted against the euro.
5. Why does the lesson convert prices through the euro?
   - (a) The ECB quotes every currency against the euro, so any pair is reached in two steps.
   - (b) The euro is the most stable currency, so the answer is the most accurate.
   - (c) Dollars cannot be converted directly into rupees under the ECB's rules.
   - (d) It avoids rounding error that a direct conversion would introduce.
6. Section 4 reads the endpoint's capacity off the benchmark at a lower concurrency than the one with the most
   output tokens a second. Why?
   - (a) The higher concurrencies were run on a different GPU, so they do not apply.
   - (b) Above it, one of this scenario's requests would take longer than the latency budget allows.
   - (c) A lower concurrency always completes more requests a second than a higher one.
   - (d) The highest throughput counts input tokens, which a request to the small model does not send.

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

A calculator whose structure outlives its prices. Later lessons in the post-training run can ask it what a
method would cost before running it: change the scenario, point it at a fresher snapshot and read the ladder.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now, so the board is current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_convert),
                              ("exercise 2", _check_request),
                              ("exercise 3", _check_fixed),
                              ("exercise 4", _check_breakeven),
                              ("exercise 5", _check_ladder)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.